# bulk COVID-19 whole blood samples

In [1]:
import lamindb as ln

ln.connect("theislab/sc-best-practices")
assert ln.setup.settings.instance.slug == "theislab/sc-best-practices"
ln.track("u1QfiYNpetF8")

→ connected lamindb: theislab/sc-best-practices
→ loaded Transform('u1QfiYNpetF80002', key='bulk_decon_covid.ipynb'), re-started Run('8qZN9LOkJGgTAHi5') at 2026-11-09 13:03:02 UTC


In [2]:
from pathlib import Path

import pandas as pd

In [3]:
import anndata as ad
import h5py
import numpy as np
import scipy.sparse as sp

h5ad_path = Path(
    "~/Downloads/seurat_COVID19_freshWB_PBMC_cohort2_incl_raw.h5ad"
).expanduser()

# The file holds whole blood and PBMC samples, with normalized values in X and raw counts in
# layers["counts"]. The bulk samples are whole blood, so the reference only needs the whole blood
# cells and their raw counts. Reading the counts in gene chunks avoids loading both full matrices.
with h5py.File(h5ad_path) as f:
    obs = ad.io.read_elem(f["obs"])
    var = ad.io.read_elem(f["var"])
    umap = ad.io.read_elem(f["obsm/X_umap"])
    keep = (obs["cells"] == "Whole_blood").to_numpy()
    counts = ad.io.sparse_dataset(
        f["layers/counts"]
    )  # stored as CSC, so column slices are cheap
    chunk = 2000
    X = sp.hstack(
        [counts[:, j : j + chunk][keep] for j in range(0, counts.shape[1], chunk)]
    ).tocsr()

assert np.array_equal(X.data, np.round(X.data)), (
    "layers['counts'] should hold integer counts"
)
X = X.astype(
    np.float32
)  # integer counts are exact in float32, at half the size of float64

adata = ad.AnnData(
    X=X,
    obs=obs[keep].copy(),
    var=pd.DataFrame(index=var.index),
    obsm={"X_umap": umap[keep]},
)
for col in adata.obs.select_dtypes("category"):
    adata.obs[col] = adata.obs[col].cat.remove_unused_categories()
adata

AnnData object with n_obs × n_vars = 89883 × 33417
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mito', 'percent.hb', 'HTO_maxID', 'HTO_secondID', 'HTO_margin', 'HTO_classification', 'HTO_classification.global', 'hash.ID', 'demultID', 'donor', 'onset_of_symptoms', 'days_after_onset', 'sampleID', 'date_of_sampling', 'experiment', 'cartridge', 'platform', 'purification', 'cells', 'age', 'sex', 'group_per_sample', 'who_per_sample', 'disease_stage', 'diagnosis', 'oxygen', 'outcome', 'comorbidities', 'COVID.19.related_medication_and_anti.microbials', 'primary_complaint', 'RNA_snn_res.0.8', 'cluster_labels_res.0.8', 'new.order', 'hpca.labels', 'blueprint.labels', 'monaco.labels', 'immune.labels', 'dmap.labels', 'hemato.labels'
    obsm: 'X_umap'
    layers: None (.X)

In [4]:
reference_path = Path("schulte_schrepping2020_covid19_wholeblood_counts.h5ad")
# The chapter environment is pinned to anndata 0.10 by rectanglepy, which cannot read the
# nullable string encoding that newer anndata versions use for pandas string columns.
with ad.settings.override(allow_write_nullable_strings=False):
    adata.write_h5ad(reference_path, compression="gzip")

ln.Artifact(
    reference_path,
    key="deconvolution/bulk_deconvolution/schulte_schrepping2020_covid19_wholeblood_counts.h5ad",
    description=(
        "scRNA-seq reference: COVID-19 fresh whole blood, cohort 2, raw counts in X, "
        "cell types in obs['cluster_labels_res.0.8'], author UMAP in obsm['X_umap']. "
        "Schulte-Schrepping et al. 2020, Cell (doi:10.1016/j.cell.2020.08.001)"
    ),
).save()

→ returning artifact with same hash: Artifact(uid='Cbed0qYVdW4lrsAy0001', key='deconvolution/bulk_deconvolution/schulte_schrepping2020_covid19_wholeblood_counts.h5ad', description='scRNA-seq reference: COVID-19 fresh whole blood, cohort 2, raw counts in X, cell types in obs['cluster_labels_res.0.8'], author UMAP in obsm['X_umap']. Schulte-Schrepping et al. 2020, Cell (doi:10.1016/j.cell.2020.08.001)', suffix='.h5ad', kind=None, otype='AnnData', size=198591867, hash='_hE_W4wRW92txt31VtzQVn', n_files=None, n_observations=None, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=224, schema_id=None, created_by_id=5, created_at=2026-11-06 13:52:43 UTC, is_locked=False, version_tag=None, is_latest=True); to track this artifact as an input, use: ln.Artifact.get()


Artifact(uid='Cbed0qYVdW4lrsAy0001', key='deconvolution/bulk_deconvolution/schulte_schrepping2020_covid19_wholeblood_counts.h5ad', description='scRNA-seq reference: COVID-19 fresh whole blood, cohort 2, raw counts in X, cell types in obs['cluster_labels_res.0.8'], author UMAP in obsm['X_umap']. Schulte-Schrepping et al. 2020, Cell (doi:10.1016/j.cell.2020.08.001)', suffix='.h5ad', kind=None, otype='AnnData', size=198591867, hash='_hE_W4wRW92txt31VtzQVn', n_files=None, n_observations=None, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=224, schema_id=None, created_by_id=5, created_at=2026-11-06 13:52:43 UTC, is_locked=False, version_tag=None, is_latest=True)

In [5]:
REPO = "https://raw.githubusercontent.com/schultzelab/COVID-19-blood-bulk-RNA-Seq/main/tables/"
GTF = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_33/gencode.v33.annotation.gtf.gz"

# --- 1. Fetch and read ---------------------------------------------------------
counts = pd.read_csv(
    REPO + "raw_counts.txt", sep="\t", index_col=0
)  # genes x samples, Ensembl IDs
counts.index.name = "gene_id"
counts.columns = counts.columns.astype(str)

metadata = pd.read_csv(REPO + "anno.txt", sep="\t", index_col=0)
metadata.index = metadata.index.astype(str)
metadata.index.name = "sample_id"

id2sym = (
    pd.read_csv(
        REPO + "ID2SYMBOL_gencode_v33.txt",
        sep="\t",
        header=None,
        index_col=0,
        usecols=[0, 1],
    )
    .iloc[:, 0]
    .rename("gene_symbol")  # without a string name, saving to parquet fails
)


# --- 2. Gene lengths from GENCODE v33 (union of exons per gene) ----------------
def gene_lengths_from_gtf(gtf_path):
    gtf = pd.read_csv(
        gtf_path,
        sep="\t",
        comment="#",
        header=None,
        usecols=[2, 3, 4, 8],
        names=["feature", "start", "end", "attr"],
    )
    ex = gtf[gtf["feature"] == "exon"].copy()
    ex["gene_id"] = ex["attr"].str.extract(r'gene_id "([^"]+)"', expand=False)
    ex = ex.sort_values(["gene_id", "start"]).reset_index(drop=True)
    prev_end = ex.groupby("gene_id")["end"].cummax().groupby(ex["gene_id"]).shift()
    block = (
        prev_end.isna() | (ex["start"] > prev_end + 1)
    ).cumsum()  # merge overlapping exons
    blocks = ex.groupby(block).agg(
        gene_id=("gene_id", "first"), start=("start", "min"), end=("end", "max")
    )
    return (
        (blocks["end"] - blocks["start"] + 1)
        .groupby(blocks["gene_id"])
        .sum()
        .rename("length_bp")
    )


lengths = gene_lengths_from_gtf(GTF)
print(
    "genes without a length:", lengths.reindex(counts.index).isna().sum()
)  # should be ~0


# --- 3. Derived tables -----------------------------------------------------------
def counts_to_tpm(counts, lengths):
    lengths = lengths.reindex(counts.index)
    keep = lengths.notna()
    rate = counts[keep].div(lengths[keep] / 1e3, axis=0)
    return rate.div(rate.sum(axis=0), axis=1) * 1e6


tpm = counts_to_tpm(counts, lengths)
bulks = (
    tpm.groupby(id2sym.reindex(tpm.index)).sum().T
)  # samples x genes, TPM (Rectangle input)
bulks.index.name = "sample_id"

metadata = metadata.loc[bulks.index]

genes without a length: 17


In [6]:
src = "Aschenbrenner et al. 2021 (doi:10.1186/s13073-020-00823-5), github.com/schultzelab/COVID-19-blood-bulk-RNA-Seq"

ln.Artifact.from_dataframe(
    bulks,
    key="deconvolution/bulk_deconvolution/covid19_wholeblood_bulk_tpm.parquet",
    description=f"Whole-blood bulk RNA-seq, TPM (exon-union lengths, GENCODE v33), samples x genes. {src}",
).save()

ln.Artifact.from_dataframe(
    metadata,
    key="deconvolution/bulk_deconvolution/covid19_wholeblood_metadata.parquet",
    description=f"Sample metadata incl. status and neutrophil counts (/mm3). {src}",
).save()

→ returning artifact with same hash: Artifact(uid='0ZMDOll0eeiu2jiB0000', key='deconvolution/bulk_deconvolution/covid19_wholeblood_bulk_tpm.parquet', description='Whole-blood bulk RNA-seq, TPM (exon-union lengths, GENCODE v33), samples x genes. Aschenbrenner et al. 2021 (doi:10.1186/s13073-020-00823-5), github.com/schultzelab/COVID-19-blood-bulk-RNA-Seq', suffix='.parquet', kind='dataset', otype='DataFrame', size=31920445, hash='BKL_r-HhLYmPnJGK2ihIxQ', n_files=None, n_observations=49, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=218, schema_id=None, created_by_id=5, created_at=2026-11-05 14:25:55 UTC, is_locked=False, version_tag=None, is_latest=True); to track this artifact as an input, use: ln.Artifact.get()
→ returning artifact with same hash: Artifact(uid='RuXh6aQaToI9tnMA0000', key='deconvolution/bulk_deconvolution/covid19_wholeblood_metadata.parquet', description='Sample metadata incl. status and neutrophil counts (/mm3). Aschenbrenner et al. 2

Artifact(uid='RuXh6aQaToI9tnMA0000', key='deconvolution/bulk_deconvolution/covid19_wholeblood_metadata.parquet', description='Sample metadata incl. status and neutrophil counts (/mm3). Aschenbrenner et al. 2021 (doi:10.1186/s13073-020-00823-5), github.com/schultzelab/COVID-19-blood-bulk-RNA-Seq', suffix='.parquet', kind='dataset', otype='DataFrame', size=14265, hash='ywXnYflpiiKFvEB4L65dTQ', n_files=None, n_observations=49, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=218, schema_id=None, created_by_id=5, created_at=2026-11-05 14:26:21 UTC, is_locked=False, version_tag=None, is_latest=True)

In [7]:
ln.finish()

• please hit CTRL + s to save the notebook in your editor .. ✓
→ finished Run('8qZN9LOkJGgTAHi5') after 44s at 2026-11-09 13:03:46 UTC
→ go to: https://lamin.ai/theislab/sc-best-practices/run/8qZN9LOkJGgTAHi5
→ to update your notebook from the CLI, run: lamin save /home/seokim/seocodes_linux/single-cell-best-practices/scripts/deconvolution/bulk_decon_covid.ipynb
